In [ ]:

from pathlib import Path
import pandas as pd

ROOT = Path("..")
RAW = ROOT / "data" / "raw" / "twcs.csv"

df_head = pd.read_csv(RAW, nrows=5)
df_head

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2.0,3
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1.0,4
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3.0,5
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4.0,6


In [ ]:
#amazon support executives replies
chunks = []
for chunk in pd.read_csv(RAW, chunksize=50_000):
    chunks.append(chunk[chunk["author_id"] == "AmazonHelp"])

df_amazon_agent = pd.concat(chunks, ignore_index=True)
print(df_amazon_agent.shape)
df_amazon_agent.head()

(169840, 7)


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,269,AmazonHelp,False,Wed Nov 22 09:23:01 +0000 2017,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...,"270,271",272.0
1,273,AmazonHelp,False,Wed Nov 22 09:40:27 +0000 2017,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...,274,271.0
2,275,AmazonHelp,False,Wed Nov 22 10:06:26 +0000 2017,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...,NaN,274.0
3,324,AmazonHelp,False,Wed Nov 22 09:06:00 +0000 2017,@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の...,NaN,325.0
4,615,AmazonHelp,False,Tue Oct 31 22:29:00 +0000 2017,@115820 I'm sorry we've let you down! Without ...,616,617.0


In [ ]:
#amazon customers queries or tweets 
chunks = []
for chunk in pd.read_csv(RAW, chunksize=50_000):
    is_agent = chunk["author_id"] == "AmazonHelp"
    is_customer = chunk["text"].str.contains("AmazonHelp", case=False, na=False)
    chunks.append(chunk[is_agent | is_customer])

df_amazon = pd.concat(chunks, ignore_index=True)
print(df_amazon.shape)
df_amazon["inbound"].value_counts()

(305019, 7)


inbound
False    169848
True     135171
Name: count, dtype: int64

In [5]:
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)

df_amazon_agent.to_parquet(OUT / "amazon_agent_tweets.parquet", index=False)
# df_amazon.to_parquet(OUT / "amazon_tweets.parquet", index=False)

In [6]:
import pandas as pd
from pathlib import Path

df = pd.read_parquet(Path("..") / "data" / "processed" / "amazon_agent_tweets.parquet")
print(df.shape)
df.head()

(169840, 7)


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,269,AmazonHelp,False,Wed Nov 22 09:23:01 +0000 2017,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...,"270,271",272.0
1,273,AmazonHelp,False,Wed Nov 22 09:40:27 +0000 2017,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...,274,271.0
2,275,AmazonHelp,False,Wed Nov 22 10:06:26 +0000 2017,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...,NaN,274.0
3,324,AmazonHelp,False,Wed Nov 22 09:06:00 +0000 2017,@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の...,NaN,325.0
4,615,AmazonHelp,False,Tue Oct 31 22:29:00 +0000 2017,@115820 I'm sorry we've let you down! Without ...,616,617.0


In [7]:
# rows that look non-ASCII / CJK
non_ascii = df_amazon_agent["text"].str.contains(r"[^\x00-\x7F]", na=False)
print("non_ascii", non_ascii.sum(), "of", len(df_amazon_agent))

non_ascii 39019 of 169840


In [8]:
# keep tweets that are mostly ASCII (English-ish). Drops JP/CN/KR etc.
ascii_ratio = df_amazon_agent["text"].fillna("").map(
    lambda t: sum(ch.isascii() for ch in t) / max(len(t), 1)
)
df_amazon_en = df_amazon_agent[ascii_ratio >= 0.9].copy()
print(df_amazon_en.shape)
df_amazon_en.head()

(159611, 7)


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
4,615,AmazonHelp,False,Tue Oct 31 22:29:00 +0000 2017,@115820 I'm sorry we've let you down! Without ...,616,617.0
5,618,AmazonHelp,False,Tue Oct 31 23:28:00 +0000 2017,@115820 We'd like to take a further look into ...,619,616.0
6,620,AmazonHelp,False,Tue Oct 31 22:28:34 +0000 2017,@115822 I am unable to affect your account via...,NaN,621.0
7,622,AmazonHelp,False,Tue Oct 31 22:28:00 +0000 2017,"@115824 Hi, wir erhalten die Filme/Serien so v...",623,624.0
8,625,AmazonHelp,False,Tue Oct 31 22:34:32 +0000 2017,@115824 Wir haben zu danken. Schönen Abend noc...,NaN,623.0


In [9]:
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)

# working copy (keep this)
df_amazon_en.to_parquet(OUT / "amazon_agent_en.parquet", index=False)

# human-readable copy
df_amazon_en.to_csv(OUT / "amazon_agent_en.csv", index=False)

print(df_amazon_en.shape)

(159611, 7)


In [10]:
from pathlib import Path
import pandas as pd

ROOT = Path("..")
PROC = ROOT / "data" / "processed"
EDA = ROOT / "data" / "eda_results"
EDA.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(PROC / "amazon_agent_en.parquet")
# if parquet missing:
# df = pd.read_csv(PROC / "amazon_agent_en.csv")

print(df.shape)
df.head(3)

(159611, 7)


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,615,AmazonHelp,False,Tue Oct 31 22:29:00 +0000 2017,@115820 I'm sorry we've let you down! Without ...,616,617.0
1,618,AmazonHelp,False,Tue Oct 31 23:28:00 +0000 2017,@115820 We'd like to take a further look into ...,619,616.0
2,620,AmazonHelp,False,Tue Oct 31 22:28:34 +0000 2017,@115822 I am unable to affect your account via...,NaN,621.0


In [11]:
print(df.dtypes)

tweet_id                     int64
author_id                      str
inbound                       bool
created_at                     str
text                           str
response_tweet_id              str
in_response_to_tweet_id    float64
dtype: object


In [12]:
print("nulls:\n", df.isna())

nulls:
         tweet_id  author_id  inbound  created_at   text  response_tweet_id  \
0          False      False    False       False  False              False   
1          False      False    False       False  False              False   
2          False      False    False       False  False               True   
3          False      False    False       False  False              False   
4          False      False    False       False  False               True   
...          ...        ...      ...         ...    ...                ...   
159606     False      False    False       False  False              False   
159607     False      False    False       False  False              False   
159608     False      False    False       False  False               True   
159609     False      False    False       False  False              False   
159610     False      False    False       False  False               True   

        in_response_to_tweet_id  
0                    

In [13]:
print("nulls sum: \n", df.isna().sum())

nulls sum: 
 tweet_id                       0
author_id                      0
inbound                        0
created_at                     0
text                           0
response_tweet_id          77346
in_response_to_tweet_id      538
dtype: int64


In [14]:
print("before", len(df))

df_clean = (
    df.dropna(subset=["tweet_id", "author_id", "text"])   # real nulls only
      .drop_duplicates(subset=["tweet_id"])               # same tweet twice
      .loc[lambda x: x["text"].astype(str).str.strip().ne("")]
      .loc[lambda x: x["author_id"].astype(str).str.strip().ne("")]
      .copy()
)

print("after", len(df_clean))
print(df_clean.isna().sum())

before 159611
after 159611
tweet_id                       0
author_id                      0
inbound                        0
created_at                     0
text                           0
response_tweet_id          77346
in_response_to_tweet_id      538
dtype: int64


In [15]:
# same Amazon copy-paste reply stored many times — keep one
n_before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["text"])
print("exact-text dups removed", n_before - len(df_clean))

exact-text dups removed 34


In [16]:
EDA = ROOT / "data" / "eda_results"
EDA.mkdir(parents=True, exist_ok=True)
df_clean.to_parquet(EDA / "amazon_agent_en_clean.parquet", index=False)
df_clean.to_csv(EDA / "amazon_agent_en_clean.csv", index=False)